# **Pipeline Pemetaan Skenario 8 – Auto-Loop Semua Kecamatan**

In [7]:
# ==============================================================
# CELL 1: PERSIAPAN LINGKUNGAN
# ==============================================================
!pip install rasterio -q
!pip install geopandas -q

import os, joblib, rasterio, numpy as np, tensorflow as tf
from scipy.signal import savgol_filter
from tensorflow.keras.models import load_model
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## **Konfigurasi Global**

In [8]:
# ==============================================================
# CELL 2: KONFIGURASI GLOBAL – Edit bagian ini sesuai kebutuhan
# ==============================================================

BASE_DIR            = '/content/drive/MyDrive/Data Skripsi Pemetaan'
NAMA_FOLDER_OUTPUT  = 'Hasil_Prediksi_Model_Skenario 9'
PATH_MODEL          = f"{BASE_DIR}/BestModel_9_Sentinel1 + Sentinel-2_Complete.keras"
PATH_SCALER         = f"{BASE_DIR}/scaler_bilstm_6class.joblib"

# Daftar semua kecamatan – tambah atau hapus nama sesuai kebutuhan
DAFTAR_KECAMATAN = [
    # "Kec_Banyuasari",
    # "Kec_Batujaya",
    # "Kec_Ciampel",
    # "Kec_Cibuaya",
    # "Kec_Cikampek",
    # "Kec_Cilamaya_Kulon",
    # "Kec_Cilamaya_Wetan",
    # "Kec_Cilebar",
    # "Kec_Jatisari",
    # "Kec_Jayakerta",
    # "Kec_Telagasari",
    # "Kec_Telukjambe_Barat",
    # "Kec_Telukjambe_Timur",
    # "Kec_Tempuran",
    "Kec_Tirtamulya",
    "Kec_Karawang_Barat",
    # "Kec_Karawang_Timur",
    # "Kec_Klari",
    # "Kec_Kota_Baru",
    # "Kec_Kutawaluya",
    # "Kec_Lemahabang",
    # "Kec_Majalaya",
    # "Kec_Pakisjaya",
    # "Kec_Pedes",
    # "Kec_Purwasari",
    # "Kec_Rawamerta",
    "Kec_Rengasdengklok",
    "Kec_Tegalwaru",
    # "Kec_Pangkalan"
]

INDEKS_HAPUS_AWAL  = 10 # Indeks variabel yang dibuang sebelum scaling (NDVI)
INDEKS_FITUR_FINAL = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]  # Fitur yang dipakai model (setelah scaling)
LABEL_PERIODE_GENAP = [2, 4, 6, 8, 10, 12]

WARNA_FASE = {
    0: (0,   0,   0,   0),   # Luar Sawah (Transparan)
    1: (144, 238, 144, 255), # Vegetatif Awal
    2: (34,  139, 34,  255), # Vegetatif Akhir
    3: (255, 255, 0,   255), # Generatif
    4: (255, 165, 0,   255), # Panen
    5: (139, 69,  19,  255), # Persiapan Lahan
    6: (210, 180, 140, 255), # Bera
}

## **Mesin Utama – Tidak Perlu Diubah**

In [9]:
# ==============================================================
# CELL 3: FUNGSI PIPELINE PREDIKSI
# ==============================================================
@tf.keras.utils.register_keras_serializable()
def subset_output(x):
    return x[:, 1::2, :]

def jalankan_pipeline_prediksi(nama_kecamatan, model, scaler):
    kec_dir   = f"{BASE_DIR}/{nama_kecamatan}"
    path_s1   = f"{kec_dir}/S1_{nama_kecamatan}_20250701_20251231.tif"
    path_x    = f"{kec_dir}/Input_BiLSTM_{nama_kecamatan}.npy"
    path_mask = f"{kec_dir}/Mask_Sawah_{nama_kecamatan}.npy"

    print(f"\n{'='*65}")
    print(f"MEMPROSES: {nama_kecamatan}")
    print(f"{'='*65}")

    try:
        print("[1] Memuat Data Numpy Mentah...")
        X_mentah    = np.load(path_x)
        mask_sawah  = np.load(path_mask)

        # Menghapus variabel yang tidak diperlukan (NDVI)
        X_12var = np.delete(X_mentah, INDEKS_HAPUS_AWAL, axis=2)
        print(f"Variabel indeks {INDEKS_HAPUS_AWAL} dihapus. Shape: {X_12var.shape}")

        print("[2] Winsorizing (3 IQR) pada 12 Variabel...")
        batas_winsor = [
            (-21.6362, 0.1891),   # Indeks 0: VV_db
            (-32.4521, -3.3976),  # Indeks 1: VH_db
            (-0.3257, 0.7346),    # Indeks 2: RPI
            (-0.0584, 1.3957),    # Indeks 3: NDPI
            (-0.0742, 0.1854),    # Indeks 4: API
            (-0.0626, 0.1768),    # Indeks 5: B2
            (-0.0385, 0.2012),    # Indeks 6: B3
            (-0.1042, 0.2577),    # Indeks 7: B4
            (-0.2071, 0.6364),    # Indeks 8: B8
            (-0.2266, 0.4776),    # Indeks 9: B11
            (-0.8752, 1.5076),    # Indeks 10: EVI
            (-0.6130, 1.1709)     # Indeks 11: LSWI
        ]

        # Eksekusi clipping pada dimensi ketiga (fitur)
        for i, (batas_bawah, batas_atas) in enumerate(batas_winsor):
            X_12var[:, :, i] = np.clip(X_12var[:, :, i], batas_bawah, batas_atas)
        print("Winsorizing selesai!")

        print("[3] Smoothing Savitzky-Golay (Window=5, Poly=2)...")
        X_smooth = savgol_filter(X_12var, window_length=5, polyorder=2, axis=1)

        print("[4] Normalisasi 12 Variabel dengan Scaler...")
        n_pixels, n_waktu, n_var = X_smooth.shape
        X_scaled = scaler.transform(X_smooth.reshape(-1, n_var)).reshape(X_smooth.shape)
        print("Berhasil terskala (0–1)!")

        print("[5] Seleksi Fitur Final...")
        X_final = X_scaled[:, :, INDEKS_FITUR_FINAL]
        print(f"Shape input model: {X_final.shape}")

        print("[5] Prediksi Bi-LSTM...")
        pred_prob  = model.predict(X_final, verbose=1)
        pred_kelas = np.argmax(pred_prob, axis=-1) + 1  # kelas 1–6

        print("[6] Ekspor GeoTIFF...")
        with rasterio.open(path_s1) as src:
            meta   = src.meta.copy()
            tinggi = src.height
            lebar  = src.width
        meta.update({'dtype': 'uint8', 'count': 1, 'nodata': 0})

        folder_out = os.path.join(kec_dir, NAMA_FOLDER_OUTPUT)
        os.makedirs(folder_out, exist_ok=True)

        for i, periode in enumerate(LABEL_PERIODE_GENAP):
            kanvas = np.zeros((tinggi, lebar), dtype='uint8')
            kanvas[mask_sawah] = pred_kelas[:, i]
            path_tif = os.path.join(folder_out, f"Pred_{nama_kecamatan}_Per_{periode:02d}.tif")
            with rasterio.open(path_tif, 'w', **meta) as dst:
                dst.write(kanvas, 1)
                dst.write_colormap(1, WARNA_FASE)

        print(f"SELESAI! Tersimpan di: {folder_out}\n")
        return True

    except Exception as e:
        print(f"GAGAL – {nama_kecamatan}: {e}\n")
        return False

## **Eksekusi – Loop Otomatis Semua Kecamatan**

In [10]:
# ==============================================================
# CELL 4: JALANKAN LOOP OTOMATIS
# – Model & scaler dimuat sekali di luar loop (efisien)
# ==============================================================

print("Memuat model dan scaler...")
model  = load_model(PATH_MODEL, custom_objects={'subset_output': subset_output})
scaler = joblib.load(PATH_SCALER)
print("Model dan scaler siap.\n")

hasil = {}
for nama_kec in DAFTAR_KECAMATAN:
    hasil[nama_kec] = jalankan_pipeline_prediksi(nama_kec, model, scaler)

# Ringkasan akhir
print("\n" + "="*65)
print("RINGKASAN HASIL EKSEKUSI")
print("="*65)
for kec, sukses in hasil.items():
    status = "Sukses" if sukses else "Gagal"
    print(f"  {status}  –  {kec}")
print("="*65)

Memuat model dan scaler...
Model dan scaler siap.


MEMPROSES: Kec_Tirtamulya
[1] Memuat Data Numpy Mentah...
Variabel indeks 10 dihapus. Shape: (356386, 12, 12)
[2] Winsorizing (3 IQR) pada 12 Variabel...
Winsorizing selesai!
[3] Smoothing Savitzky-Golay (Window=5, Poly=2)...
[4] Normalisasi 12 Variabel dengan Scaler...


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


Berhasil terskala (0–1)!
[5] Seleksi Fitur Final...
Shape input model: (356386, 12, 12)
[5] Prediksi Bi-LSTM...
11138/11138 ━━━━━━━━━━━━━━━━━━━━ 122s 11ms/step
[6] Ekspor GeoTIFF...
SELESAI! Tersimpan di: /content/drive/MyDrive/Data Skripsi Pemetaan/Kec_Tirtamulya/Hasil_Prediksi_Model_Skenario 9


MEMPROSES: Kec_Karawang_Barat
[1] Memuat Data Numpy Mentah...
Variabel indeks 10 dihapus. Shape: (192414, 12, 12)
[2] Winsorizing (3 IQR) pada 12 Variabel...
Winsorizing selesai!
[3] Smoothing Savitzky-Golay (Window=5, Poly=2)...
[4] Normalisasi 12 Variabel dengan Scaler...


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


Berhasil terskala (0–1)!
[5] Seleksi Fitur Final...
Shape input model: (192414, 12, 12)
[5] Prediksi Bi-LSTM...
6013/6013 ━━━━━━━━━━━━━━━━━━━━ 68s 11ms/step
[6] Ekspor GeoTIFF...
SELESAI! Tersimpan di: /content/drive/MyDrive/Data Skripsi Pemetaan/Kec_Karawang_Barat/Hasil_Prediksi_Model_Skenario 9


MEMPROSES: Kec_Rengasdengklok
[1] Memuat Data Numpy Mentah...
Variabel indeks 10 dihapus. Shape: (204587, 12, 12)
[2] Winsorizing (3 IQR) pada 12 Variabel...
Winsorizing selesai!
[3] Smoothing Savitzky-Golay (Window=5, Poly=2)...
[4] Normalisasi 12 Variabel dengan Scaler...
Berhasil terskala (0–1)!
[5] Seleksi Fitur Final...


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


Shape input model: (204587, 12, 12)
[5] Prediksi Bi-LSTM...
6394/6394 ━━━━━━━━━━━━━━━━━━━━ 70s 11ms/step
[6] Ekspor GeoTIFF...
SELESAI! Tersimpan di: /content/drive/MyDrive/Data Skripsi Pemetaan/Kec_Rengasdengklok/Hasil_Prediksi_Model_Skenario 9


MEMPROSES: Kec_Tegalwaru
[1] Memuat Data Numpy Mentah...
Variabel indeks 10 dihapus. Shape: (225460, 12, 12)
[2] Winsorizing (3 IQR) pada 12 Variabel...
Winsorizing selesai!
[3] Smoothing Savitzky-Golay (Window=5, Poly=2)...
[4] Normalisasi 12 Variabel dengan Scaler...
Berhasil terskala (0–1)!
[5] Seleksi Fitur Final...


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


Shape input model: (225460, 12, 12)
[5] Prediksi Bi-LSTM...
7046/7046 ━━━━━━━━━━━━━━━━━━━━ 76s 11ms/step
[6] Ekspor GeoTIFF...
SELESAI! Tersimpan di: /content/drive/MyDrive/Data Skripsi Pemetaan/Kec_Tegalwaru/Hasil_Prediksi_Model_Skenario 9


RINGKASAN HASIL EKSEKUSI
  Sukses  –  Kec_Tirtamulya
  Sukses  –  Kec_Karawang_Barat
  Sukses  –  Kec_Rengasdengklok
  Sukses  –  Kec_Tegalwaru
